# Construct BOW, DTM, TFIDF, TFIDF_L2

BOW → DTM → TFIDF → TFIDF_L2, plus the DFIDF computation back-joined to VOCAB.

This is also you declare your bag level and your significant-word selection principle for TFIDF_L2

Note: I have 22 book_ids but 12 true novels. The stories from the collection "poirot investigates" are split into individual book_ids.

I am choosing Chapter as my bag because it is a good narrative WORD that isn't too big...

I am following along with my M07 HW as I complete this section.

## Setup

### Import Libraries

In [1]:
# import libraries
import pandas as pd
import numpy as np
from numpy.linalg import norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer

### Configuration

In [2]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [3]:
# set chapter as bag
bag = "CHAPS"

### Load Data

In [4]:
# load in tables
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)
VOCAB = pd.read_csv('data/VOCAB.csv', sep='\t').set_index('term_str')

## Build BOW

Create a BOW with chapter as the bag.

In [5]:
# insert corpus_to_bow function from M05 Homework
def corpus_to_bow(df=None, bag='CHAPS'):
    '''
    Function that generates a bag of words BOW representation of the CORPUS table.
    Arguments: 
    - Tokens dataframe. Default is entire CORPUS table but can be a subset of it.
    - Choice of bag, i.e. `OHCO` level, such as book, chapter, or paragraph. Default is "CHAPS"
    Output: 
    - BOW table: A multiindex (bag levels + term_str) dataframe and a column count 'n'
    '''
    if df is None:
        raise ValueError("Must pass a DataFrame")
    
    # grab column names for bag
    bag_cols = bags[bag]
    
    BOW = df.groupby(bag_cols + ['term_str']).term_str.count().to_frame('n')

    return BOW

In [6]:
# create BOW with chapter as bag
BOW_chaps = corpus_to_bow(CORPUS, bag)

BOW_chaps

n
book_id                      chap_num term_str      
giants-bread                 0        a           53
                                      about        3
                                      above        1
                                      absolute     1
                                      absolutely   1
...                                               ..
the-tragedy-at-marsdon-manor 1        yielded      1
                                      you         79
                                      young       11
                                      your        10
                                      yourself     1

[252614 rows x 1 columns]

## Build DTM

Unstack the BOW into DTM and replace nulls with 0s.

In [7]:
# unstack BOW into DTM and replace nulls with 0s
DTM = BOW_chaps.unstack(fill_value=0)

# drop the 'n' top level column artifact that came over from BOW count
DTM.columns = DTM.columns.droplevel(0)

DTM

term_str                               1  10  100  100000  1015  1019  102  \
book_id                      chap_num                                        
giants-bread                 0         0   0    0       0     0     0    0   
                             1         1   0    0       0     0     0    0   
                             2         0   0    0       0     0     0    0   
                             3         0   0    0       0     0     0    0   
                             4         0   0    0       0     0     0    0   
...                                   ..  ..  ...     ...   ...   ...  ...   
the-seven-dials-mystery      31        0   0    0       0     0     0    0   
                             32        3   0    0       0     0     0    0   
                             33        0   0    0       0     0     0    0   
                             34        0   0    0       0     0     0    0   
the-tragedy-at-marsdon-manor 1         0   0    0       0     0     0    0   

term_str                               1023  103  1030  ...  épatant  \
book_id                      chap_num                   ...            
giants-bread                 0            0    0     0  ...        0   
                             1            0    0     0  ...        0   
                             2            0    0     0  ...        0   
                             3            0    0     0  ...        0   
                             4            0    0     0  ...        0   
...                                     ...  ...   ...  ...      ...   
the-seven-dials-mystery      31           0    0     0  ...        0   
                             32           0    0     0  ...        0   
                             33           0    0     0  ...        0   
                             34           0    0     0  ...        0   
the-tragedy-at-marsdon-manor 1            0    0     0  ...        0   

term_str                               épatants  épouvantable  état  étre  \
book_id                      chap_num                                       
giants-bread                 0                0             0     0     0   
                             1                0             0     0     0   
                             2                0             0     0     0   
                             3                0             0     0     0   
                             4                0             0     0     0   
...                                         ...           ...   ...   ...   
the-seven-dials-mystery      31               0             0     0     0   
                             32               0             0     0     0   
                             33               0             0     0     0   
                             34               0             0     0     0   
the-tragedy-at-marsdon-manor 1                0             0     0     0   

term_str                               évidemment  évident  êtes  œuvre  ⁷₁₁  
book_id                      chap_num                                         
giants-bread                 0                  0        0     0      0    0  
                             1                  0        0     0      0    0  
                             2                  0        0     0      0    0  
                             3                  0        0     0      0    0  
                             4                  0        0     0      0    0  
...                                           ...      ...   ...    ...  ...  
the-seven-dials-mystery      31                 0        0     0      0    0  
                             32                 0        0     0      0    0  
                             33                 0        0     0      0    0  
                             34                 0        0     0      0    0  
the-tragedy-at-marsdon-manor 1                  0        0     0      0    0  

[325 rows x 20953 

## Build TFIDF

Use SKLearn to convert DTM into a TFIDF data frame by passing DTM to TfidfTransformer with defaults and then saving the results to a dataframe TFIDF that preserves the index and columns of DTM.

In [8]:
# instantiate tfidf engine with norm=None to get raw TFIDF
tfidf_engine = TfidfTransformer(norm=None)

# fit and transform the DTM and then convert output back into a dataframe
TFIDF = pd.DataFrame(
    tfidf_engine.fit_transform(DTM).toarray(), #TfidfTransformer() ouputs a sparse matrix so we use .toarray() to convert it to a dense one
    index = DTM.index,
    columns = DTM.columns
)

TFIDF

term_str                                       1   10  100  100000  1015  \
book_id                      chap_num                                      
giants-bread                 0          0.000000  0.0  0.0     0.0   0.0   
                             1          3.568022  0.0  0.0     0.0   0.0   
                             2          0.000000  0.0  0.0     0.0   0.0   
                             3          0.000000  0.0  0.0     0.0   0.0   
                             4          0.000000  0.0  0.0     0.0   0.0   
...                                          ...  ...  ...     ...   ...   
the-seven-dials-mystery      31         0.000000  0.0  0.0     0.0   0.0   
                             32        10.704065  0.0  0.0     0.0   0.0   
                             33         0.000000  0.0  0.0     0.0   0.0   
                             34         0.000000  0.0  0.0     0.0   0.0   
the-tragedy-at-marsdon-manor 1          0.000000  0.0  0.0     0.0   0.0   

term_str                               1019  102  1023  103  1030  ...  \
book_id                      chap_num                              ...   
giants-bread                 0          0.0  0.0   0.0  0.0   0.0  ...   
                             1          0.0  0.0   0.0  0.0   0.0  ...   
                             2          0.0  0.0   0.0  0.0   0.0  ...   
                             3          0.0  0.0   0.0  0.0   0.0  ...   
                             4          0.0  0.0   0.0  0.0   0.0  ...   
...                                     ...  ...   ...  ...   ...  ...   
the-seven-dials-mystery      31         0.0  0.0   0.0  0.0   0.0  ...   
                             32         0.0  0.0   0.0  0.0   0.0  ...   
                             33         0.0  0.0   0.0  0.0   0.0  ...   
                             34         0.0  0.0   0.0  0.0   0.0  ...   
the-tragedy-at-marsdon-manor 1          0.0  0.0   0.0  0.0   0.0  ...   

term_str                               épatant  épatants  épouvantable  état  \
book_id                      chap_num                                          
giants-bread                 0             0.0       0.0           0.0   0.0   
                             1             0.0       0.0           0.0   0.0   
                             2             0.0       0.0           0.0   0.0   
                             3             0.0       0.0           0.0   0.0   
                             4             0.0       0.0           0.0   0.0   
...                                        ...       ...           ...   ...   
the-seven-dials-mystery      31            0.0       0.0           0.0   0.0   
                             32            0.0       0.0           0.0   0.0   
                             33            0.0       0.0           0.0   0.0   
                             34            0.0       0.0           0.0   0.0   
the-tragedy-at-marsdon-manor 1             0.0       0.0           0.0   0.0   

term_str                               étre  évidemment  évident  êtes  œuvre  \
book_id                      chap_num                                           
giants-bread                 0          0.0         0.0      0.0   0.0    0.0   
                             1          0.0         0.0      0.0   0.0    0.0   
                             2          0.0         0.0      0.0   0.0    0.0   
                             3          0.0         0.0      0.0   0.0    0.0   
                             4          0.0         0.0      0.0   0.0    0.0   
...                                     ...         ...      ...   ...    ...   
the-seven-dials-mystery      31         0.0         0.0      0.0   0.0    0.0   
                             32         0.0         0.0      0.0   0.0    0.0   
                             33         0.0         0.0      0.0   0.0    0.0   
                             34         0.0         0.0      0.0   0.0    0.0   
the-tragedy-at-marsdon-manor 1     

## Reduce Feature Space (and compute DFIDF)

Reduce the feature space of DTM, i.e. the vocabulary, by filtering for words whose document frequency is greater than or equal to 50 and information is greater than or equal to 10.

I have chosen these same thresholds as the M07 HW to start with because there were 320 documents in that corpus and I have 325 in mine.

In [9]:
# calculate DF (document frequency)
df_count = (DTM > 0).sum()

# calculate I (information)
tf = DTM.sum()
p = tf / tf.sum()
information = -np.log2(p)

In [10]:
# create term stats dataframe
TERM_STATS = pd.DataFrame({'df': df_count, 'i': information})

TERM_STATS

,df,i
term_str,,
1,24,14.656108
10,2,18.700502
100,2,18.700502
100000,1,19.700502
1015,1,19.700502
...,...,...
évidemment,3,18.115540
évident,1,19.700502
êtes,1,19.700502


In [11]:
# get index to filter for df_count >= 50 and i >= 10
keep_terms = TERM_STATS[(TERM_STATS['df'] >= 50) & (TERM_STATS['i'] >= 10)].index

keep_terms

Index(['able', 'above', 'abruptly', 'absolutely', 'absurd', 'accepted',
       'account', 'across', 'act', 'actually',
       ...
       'wrong', 'wrote', 'yard', 'year', 'years', 'yesterday', 'yet', 'young',
       'yours', 'yourself'],
      dtype='str', name='term_str', length=961)

In [12]:
# apply filter to TFIDF table
TFIDF_reduced = TFIDF[keep_terms]

TFIDF_reduced

term_str                                   able     above  abruptly  \
book_id                      chap_num                                 
giants-bread                 0         0.000000  2.779564  0.000000   
                             1         0.000000  2.779564  0.000000   
                             2         0.000000  0.000000  0.000000   
                             3         0.000000  2.779564  0.000000   
                             4         3.565902  0.000000  2.779564   
...                                         ...       ...       ...   
the-seven-dials-mystery      31        1.782951  2.779564  0.000000   
                             32        0.000000  0.000000  2.779564   
                             33        5.348853  0.000000  0.000000   
                             34        0.000000  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1         1.782951  0.000000  0.000000   

term_str                               absolutely    absurd  accepted  \
book_id                      chap_num                                   
giants-bread                 0           2.191778  2.538402  0.000000   
                             1           0.000000  0.000000  2.835654   
                             2           0.000000  0.000000  2.835654   
                             3           0.000000  0.000000  0.000000   
                             4           2.191778  5.076804  2.835654   
...                                           ...       ...       ...   
the-seven-dials-mystery      31          2.191778  0.000000  0.000000   
                             32          0.000000  0.000000  0.000000   
                             33          2.191778  0.000000  0.000000   
                             34          0.000000  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1           4.383555  0.000000  2.835654   

term_str                               account    across       act  actually  \
book_id                      chap_num                                          
giants-bread                 0             0.0  1.668904  0.000000  0.000000   
                             1             0.0  1.668904  0.000000  0.000000   
                             2             0.0  3.337807  0.000000  0.000000   
                             3             0.0  0.000000  0.000000  0.000000   
                             4             0.0  0.000000  0.000000  2.298261   
...                                        ...       ...       ...       ...   
the-seven-dials-mystery      31            0.0  0.000000  0.000000  0.000000   
                             32            0.0  1.668904  0.000000  0.000000   
                             33            0.0  0.000000  0.000000  9.193044   
                             34            0.0  0.000000  0.000000  2.298261   
the-tragedy-at-marsdon-manor 1             0.0  1.668904  5.633211  0.000000   

term_str                               ...     wrong     wrote  yard  \
book_id                      chap_num  ...                             
giants-bread                 0         ...  1.789685  0.000000   0.0   
                             1         ...  0.000000  0.000000   0.0   
                             2         ...  3.579370  0.000000   0.0   
                             3         ...  0.000000  0.000000   0.0   
                             4         ...  0.000000  0.000000   0.0   
...                                    ...       ...       ...   ...   
the-seven-dials-mystery      31        ...  0.000000  0.000000   0.0   
                             32        ...  0.000000  0.000000   0.0   
                             33        ...  1.789685  2.496438   0.0   
                             34        ...  0.000000  0.000000   0.0   
the-tragedy-at-marsdon-manor 1         ...  0.000000  0.000000   0.0   

term_str                                   year     years  yesterday  \
book_id                      chap_num                                  
gian

## Normalize (TFIDF_L2)

L2 normalize TFIDF_reduced. 

L2 normalization is applied after feature space reduction so that row vector norms reflect only the significant term dimensions that PCA will operate on.

To normalize a vector, we divide each element by the length of its norm $L_p$. Length measures vary by $p$. In the case of L2 normaliztion, $L_2$ is the square root of the sum of each element squared.

In [13]:
TFIDF_L2 = TFIDF_reduced.apply(lambda x: x / norm(x), axis=1) # Pythagorean, AKA Euclidean

TFIDF_L2

term_str                                   able     above  abruptly  \
book_id                      chap_num                                 
giants-bread                 0         0.000000  0.055211  0.000000   
                             1         0.000000  0.033963  0.000000   
                             2         0.000000  0.000000  0.000000   
                             3         0.000000  0.029749  0.000000   
                             4         0.031875  0.000000  0.024846   
...                                         ...       ...       ...   
the-seven-dials-mystery      31        0.020911  0.032600  0.000000   
                             32        0.000000  0.000000  0.068863   
                             33        0.057557  0.000000  0.000000   
                             34        0.000000  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1         0.015280  0.000000  0.000000   

term_str                               absolutely    absurd  accepted  \
book_id                      chap_num                                   
giants-bread                 0           0.043536  0.050421  0.000000   
                             1           0.000000  0.000000  0.034649   
                             2           0.000000  0.000000  0.053847   
                             3           0.000000  0.000000  0.000000   
                             4           0.019592  0.045381  0.025348   
...                                           ...       ...       ...   
the-seven-dials-mystery      31          0.025706  0.000000  0.000000   
                             32          0.000000  0.000000  0.000000   
                             33          0.023585  0.000000  0.000000   
                             34          0.000000  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1           0.037567  0.000000  0.024302   

term_str                               account    across       act  actually  \
book_id                      chap_num                                          
giants-bread                 0             0.0  0.033150  0.000000  0.000000   
                             1             0.0  0.020392  0.000000  0.000000   
                             2             0.0  0.063383  0.000000  0.000000   
                             3             0.0  0.000000  0.000000  0.000000   
                             4             0.0  0.000000  0.000000  0.020544   
...                                        ...       ...       ...       ...   
the-seven-dials-mystery      31            0.0  0.000000  0.000000  0.000000   
                             32            0.0  0.041347  0.000000  0.000000   
                             33            0.0  0.000000  0.000000  0.098922   
                             34            0.0  0.000000  0.000000  0.078589   
the-tragedy-at-marsdon-manor 1             0.0  0.014303  0.048277  0.000000   

term_str                               ...     wrong     wrote  yard  \
book_id                      chap_num  ...                             
giants-bread                 0         ...  0.035549  0.000000   0.0   
                             1         ...  0.000000  0.000000   0.0   
                             2         ...  0.067970  0.000000   0.0   
                             3         ...  0.000000  0.000000   0.0   
                             4         ...  0.000000  0.000000   0.0   
...                                    ...       ...       ...   ...   
the-seven-dials-mystery      31        ...  0.000000  0.000000   0.0   
                             32        ...  0.000000  0.000000   0.0   
                             33        ...  0.019258  0.026863   0.0   
                             34        ...  0.000000  0.000000   0.0   
the-tragedy-at-marsdon-manor 1         ...  0.000000  0.000000   0.0   

term_str                                   year     years  yesterday  \
book_id                      chap_num                                  
gian

## Update VOCAB with DFIDF

In [ ]:
# add df column to VOCAB
VOCAB = VOCAB.join(TERM_STATS[['df']]) # bring over df (and only df) from TERM_STATS (join merges on index (term_str) by default)

# compute and add idf to VOCAB (recall idf=log2(N/df) where N is number of bags)
N = CORPUS.groupby(bags[bag]).ngroups
VOCAB['idf'] = np.log2(N/VOCAB['df'])

# compute and add dfidf VOCAB
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['idf'] # DFIDF is the same as mean boolean TFIDF (doc freq * inv doc freq)

# VOCAB

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos,stop,porter_stem,df,idf,dfidf
term_str,,,,,,,,,,,,,
1,33,1,0.000039,14.656108,CD,CD,1,1,False,1,24,3.759333,90.224002
10,2,2,0.000002,18.700502,CD,CD,1,1,False,10,2,7.344296,14.688592
100,2,3,0.000002,18.700502,CD,VB,2,2,False,100,2,7.344296,14.688592
100000,1,6,0.000001,19.700502,NN,NN,1,1,False,100000,1,8.344296,8.344296
1015,1,4,0.000001,19.700502,CD,CD,1,1,False,1015,1,8.344296,8.344296
...,...,...,...,...,...,...,...,...,...,...,...,...,...
évidemment,3,10,0.000004,18.115540,NN,NN,1,1,False,évidem,3,6.759333,20.278000
évident,1,7,0.000001,19.700502,NN,NN,1,1,False,évident,1,8.344296,8.344296
êtes,1,4,0.000001,19.700502,NNS,NN,1,1,False,ête,1,8.344296,8.344296


## Save Outputs

In [ ]:
# save the updated VOCAB table to csv (replace the existing one created by 03-VOCAB.ipynb)
VOCAB.to_csv('data/VOCAB.csv', sep='\t', index=True)

# save the BOW table to csv
BOW_chaps.to_csv('data/BOW_chaps.csv', sep='\t', index=True)

# save the DTM table to csv
DTM.to_csv('data/DTM.csv', sep='\t', index=True)

# save the TFIDF table to csv
TFIDF.to_csv('data/TFIDF.csv', sep='\t', index=True)

# save the TFIDF_L2 table to csv
TFIDF_L2.to_csv('data/TFIDF_L2.csv', sep='\t', index=True)